# Synthetic Tweet Generation — API Sweep (OpenAI + Anthropic)

Generates synthetic data from **multiple API-based LLMs** (OpenAI + Anthropic) sequentially. Same prompt grid, same `TWEETS_PER_CELL`, same `SEED_BASE` as `01b_DataGenerator_HuggingFace.ipynb` — outputs are directly comparable across all generators.

**Designed for server overnight runs.** No GPU needed. Streaming chat completions are rate-limit friendly and inherently sequential, so this notebook is happy to run unattended via `nohup`/`tmux`.

**Workflow:**
1. Edit `MODEL_LIST` to include the API models you want
2. Run the smoke test to verify keys + reachable endpoints
3. Set `TWEETS_PER_CELL = 1200`, kick off, walk away

**Provider dispatch:** each entry is a `(provider, model_id)` tuple. The runner picks the right SDK at call time. Currently supports `openai` and `anthropic`.

**Cost envelope (full 1,200/cell run, batch tier where available, ~$0.0X per 1k tokens):**
- `gpt-4o-mini`: ~$0.30
- `gpt-5.4-mini`: ~$3.65
- `claude-haiku-4-5`: ~$5–8
- (Verify on OpenAI/Anthropic pricing page before running — pricing changes.)

**Output:** one CSV per model in `data/synthetic_data/{model-tag}_synthetic_{N}per_cell_{ts}.csv`.

## 1. Environment + API clients

In [ ]:
import os, re, time, json
from datetime import datetime
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from tqdm.auto import tqdm

load_dotenv()

# --- OpenAI (always required for OpenAI models) ---
from openai import OpenAI
openai_client = None
if os.getenv('OPENAI_API_KEY'):
    openai_client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))
    print('OpenAI client : ready')
else:
    print('OpenAI client : NO KEY (set OPENAI_API_KEY in .env)')

# --- Anthropic (lazy import — only fails if you actually use a Claude model without the package) ---
anthropic_client = None
try:
    from anthropic import Anthropic
    if os.getenv('ANTHROPIC_API_KEY'):
        anthropic_client = Anthropic(api_key=os.getenv('ANTHROPIC_API_KEY'))
        print('Anthropic client: ready')
    else:
        print('Anthropic client: NO KEY (set ANTHROPIC_API_KEY in .env if you want Claude models)')
except ImportError:
    print('Anthropic SDK not installed. To enable Claude:  pip install anthropic')


## 2. Run configuration

Edit `MODEL_LIST` to add/remove generators. Each entry is `(provider, model_id)`. Comment out a line to skip.

In [ ]:
MODEL_LIST = [
    # ('openai',    'gpt-4o-mini-2024-07-18'),  # ALREADY DONE — keep commented out to avoid re-spending
    ('openai',    'gpt-5.4-mini'),                # newer OpenAI alignment generation
    ('anthropic', 'claude-haiku-4-5'),            # Anthropic RLHF — different commercial regime
    # ('openai',    'gpt-4-turbo'),               # uncomment if you have budget headroom
    # ('anthropic', 'claude-sonnet-4-5'),         # bigger Claude — much more expensive
]

TWEETS_PER_CELL = 25       # 25 = pilot, 1200 = full
SEED_BASE       = 42

TARGETS = ['Donald Trump', 'Joe Biden', 'Bernie Sanders']
STANCES = ['FAVOR', 'AGAINST']

OUTPUT_DIR = Path('data/synthetic_data')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Models to run   : {len(MODEL_LIST)}')
for i, (p, m) in enumerate(MODEL_LIST, 1):
    print(f'  {i}. [{p}] {m}')
print(f'Tweets per cell : {TWEETS_PER_CELL}')
print(f'Total cells     : {len(TARGETS) * len(STANCES)}')


## 3. Prompt grid (identical to `01b_DataGenerator_HuggingFace.ipynb`)

In [ ]:
SYSTEM_PROMPT = (
    'You are a creative-writing assistant generating fictional social-media '
    'posts for an academic NLP research dataset. The posts are training data '
    'for a stance-classification model — they are not real opinions and will '
    'not be published as real tweets. '
    'Real tweets are short (often 80–160 characters), informal, conversational, '
    'and may include hashtags or casual phrasing. '
    'Output only the post text. No quotes, no preamble, no caveats, no disclaimers.'
)

TEMPLATES = [
    'Write a single tweet that expresses a stance {direction} {target}, focused on {topic}. Format it as {fmt}. Stay under 200 characters.',
    'Compose one tweet from a regular Twitter user whose stance is {direction} {target}. The tweet is about {topic} and reads like {fmt}. Maximum 200 characters.',
    'Generate a realistic tweet expressing a position {direction} {target}. The subject is {topic} and the tone should fit {fmt}. Keep under 200 characters.',
    'Produce a single short tweet — stance: {direction} {target}, topic: {topic}, format: {fmt}. Real Twitter user voice. Under 200 characters.',
]
TOPICS = [
    'their record in office or public life',
    'their position on a policy issue',
    'the campaign trail',
    'a recent news cycle moment',
    'the primary race',
    'the upcoming general election',
]
FORMATS = [
    'a brief personal reaction',
    'a one-line observation',
    'a rhetorical question',
    'a sharp one-liner',
]

def build_prompt(target, stance, idx):
    direction = 'in favor of' if stance == 'FAVOR' else 'against'
    t  = TEMPLATES[idx % len(TEMPLATES)]
    tp = TOPICS[(idx // len(TEMPLATES)) % len(TOPICS)]
    fm = FORMATS[(idx // (len(TEMPLATES) * len(TOPICS))) % len(FORMATS)]
    return t.format(direction=direction, target=target, topic=tp, fmt=fm)

REFUSAL_HEADS = ("i can't", "i cannot", "i'm sorry", 'as an ai',
                 "i won't", "i'm not able", "i'm unable")

def is_refusal(text):
    if not text or len(text) < 20:
        return True
    return text.lower().lstrip('"\' ').startswith(REFUSAL_HEADS)

def normalize(text):
    return re.sub(r'[^\w\s]', '', text.lower()).strip() if text else ''

for i in range(3):
    print(f'[{i}] {build_prompt("Donald Trump", "FAVOR", i)}')

## 4. Provider dispatch

One function per provider. Both return a single string (the model's output text). Reasoning models (GPT-5 family) get `reasoning_effort` and `max_completion_tokens` automatically; non-reasoning models get `temperature` + `max_tokens`.

In [ ]:
def call_openai(model, system, user, seed):
    """Single OpenAI call. Auto-handles reasoning vs non-reasoning models."""
    if openai_client is None:
        raise RuntimeError('OPENAI_API_KEY not set')

    is_reasoning = model.startswith('gpt-5') or model.startswith('o1') or model.startswith('o3') or model.startswith('o4')
    kwargs = {
        'model': model,
        'messages': [
            {'role': 'system', 'content': system},
            {'role': 'user',   'content': user},
        ],
        'seed': seed,
    }
    if is_reasoning:
        # GPT-5.4 family uses 'none'; GPT-5.0 family uses 'minimal'
        kwargs['reasoning_effort'] = 'none' if any(v in model for v in ('5.4', '5.5')) else 'minimal'
        kwargs['max_completion_tokens'] = 2000   # reasoning eats internal tokens
    else:
        kwargs['temperature'] = 0.9
        kwargs['max_tokens']  = 70

    resp = openai_client.chat.completions.create(**kwargs)
    return (resp.choices[0].message.content or '').strip().strip('"\'')

def call_anthropic(model, system, user, seed):
    """Single Anthropic call. Anthropic API doesn't accept `seed`; sampling reproducibility is best-effort."""
    if anthropic_client is None:
        raise RuntimeError('ANTHROPIC_API_KEY not set or anthropic SDK not installed')
    resp = anthropic_client.messages.create(
        model       = model,
        max_tokens  = 120,
        temperature = 0.9,
        system      = system,
        messages    = [{'role': 'user', 'content': user}],
    )
    text = ''.join(block.text for block in resp.content if hasattr(block, 'text'))
    return text.strip().strip('"\'')

DISPATCH = {
    'openai':    call_openai,
    'anthropic': call_anthropic,
}

# Smoke test for each provider that has a model in the list
providers_used = sorted({p for p, _ in MODEL_LIST})
for p in providers_used:
    sample_model = next(m for prov, m in MODEL_LIST if prov == p)
    try:
        text = DISPATCH[p](sample_model, SYSTEM_PROMPT, build_prompt('Donald Trump', 'FAVOR', 0), seed=SEED_BASE)
        print(f'[{p}] {sample_model}\n   → {text[:160]}\n')
    except Exception as e:
        print(f'[{p}] {sample_model}\n   ✗ {str(e)[:200]}\n')

## 5. Per-model generation function

Same dedup-and-retry loop as the HF notebook. Saves an incremental progress CSV per cell so a crash mid-cell doesn't lose work.

In [ ]:
def run_one_api(provider, model_id, tweets_per_cell):
    print(f'\n{"=" * 60}\n  [{provider}] {model_id}\n{"=" * 60}')
    call_fn = DISPATCH[provider]
    rows = []
    t0 = time.time()

    for ti, target in enumerate(TARGETS):
        for si, stance in enumerate(STANCES):
            offset    = (ti * len(STANCES) + si) * 100_000
            seen_norm = set()
            kept      = 0
            idx       = 0
            max_tries = tweets_per_cell * 2

            pbar = tqdm(total=tweets_per_cell, desc=f'{target}/{stance}', leave=False)
            while kept < tweets_per_cell and idx < max_tries:
                seed = SEED_BASE + offset + idx
                try:
                    text = call_fn(model_id, SYSTEM_PROMPT, build_prompt(target, stance, idx), seed=seed)
                    err  = None
                except Exception as e:
                    text, err = None, str(e)

                refused = is_refusal(text)
                norm    = normalize(text) if text else ''

                if not refused and not err and norm and norm not in seen_norm:
                    seen_norm.add(norm)
                    rows.append({
                        'Tweet': text, 'Target': target, 'Stance': stance,
                        'model': model_id, 'provider': provider,
                        'prompt_idx': idx, 'seed': seed,
                        'refused': False, 'error': None,
                    })
                    kept += 1
                    pbar.update(1)
                idx += 1
            pbar.close()
            print(f'  {target}/{stance}: kept {kept}/{tweets_per_cell} after {idx} tries')

            tag = model_id.replace('/', '_')
            pd.DataFrame(rows).to_csv(OUTPUT_DIR / f'{tag}_progress.csv', index=False)

    print(f'  Run finished in {time.time()-t0:.0f}s')
    return pd.DataFrame(rows)

## 6. Run all API models

In [ ]:
results   = {}   # model_id -> DataFrame
csv_paths = {}   # model_id -> Path

for provider, model_id in MODEL_LIST:
    try:
        df = run_one_api(provider, model_id, tweets_per_cell=TWEETS_PER_CELL)
    except Exception as e:
        print(f'\n  [SKIPPED] {model_id} — {str(e)[:200]}')
        continue

    ts  = datetime.now().strftime('%Y%m%d_%H%M%S')
    tag = model_id.replace('/', '_')
    path = OUTPUT_DIR / f'{tag}_synthetic_{TWEETS_PER_CELL}per_cell_{ts}.csv'
    df.to_csv(path, index=False)
    print(f'  ✅ Wrote {len(df)} rows → {path}')

    results[model_id]   = df
    csv_paths[model_id] = path

print(f'\n{"=" * 60}')
print(f'  Done. {len(results)}/{len(MODEL_LIST)} models successful.')
print(f'{"=" * 60}')

## 7. Quick comparison

Headline metrics per model: char length, hashtag rate, top-trigram concentration, uniqueness. Use this to decide which 2–3 API models to scale up to full 1,200/cell, alongside the 2–3 best HF models from `01b`.

In [ ]:
from collections import Counter
import re as _re

def quick_summary(df):
    df = df.copy()
    df['char_len']   = df['Tweet'].str.len()
    df['n_hashtags'] = df['Tweet'].str.count(r'#\w+')
    df['_norm']      = df['Tweet'].apply(normalize)
    top_tri_pct = []
    for (t, s), grp in df.groupby(['Target', 'Stance']):
        c = Counter()
        for tw in grp['Tweet']:
            words = _re.findall(r'[a-z]+', str(tw).lower())
            for i in range(len(words) - 2):
                c[(words[i], words[i+1], words[i+2])] += 1
        if c:
            top_tri_pct.append(c.most_common(1)[0][1] / len(grp))
    return {
        'n':                    len(df),
        'char_len':             round(df['char_len'].mean(), 1),
        'n_hashtags':           round(df['n_hashtags'].mean(), 2),
        'unique_pct':           round(df['_norm'].nunique() / len(df) * 100, 1),
        'top_trigram_pct_avg':  round(sum(top_tri_pct)/len(top_tri_pct) * 100, 1) if top_tri_pct else 0.0,
    }

rows = []
for model_id, df in results.items():
    s = quick_summary(df)
    s['model'] = model_id
    rows.append(s)

leaderboard = pd.DataFrame(rows).set_index('model')[
    ['n', 'char_len', 'n_hashtags', 'unique_pct', 'top_trigram_pct_avg']
]
print('Quick-comparison leaderboard:')
print(leaderboard)

print('\n3 samples per model (Trump-FAVOR):')
for model_id, df in results.items():
    print(f'\n--- {model_id} ---')
    sub = df[(df.Target == 'Donald Trump') & (df.Stance == 'FAVOR')]
    for tw in sub['Tweet'].head(3):
        print(f'  • {tw[:180]}')

## 8. Next steps

1. **Eyeball the leaderboard.** API models typically have similar form factor (~170 chars, 1.5+ hashtags). Look for divergence in trigram concentration and uniqueness — those are the alignment-regime fingerprints.
2. **Sync CSVs back to local repo:** all `data/synthetic_data/*_synthetic_*per_cell_*.csv` files.
3. **Audit:** open `03_SyntheticAudit.ipynb`, add the new CSV paths to the `SYN_PATHS` dict (cell 2). Everything else iterates automatically.
4. **Decide which to scale.** Pick the most-distinctive 2–3 API models, set `TWEETS_PER_CELL = 1200`, kick off overnight on the server.
5. **Server invocation:**
   ```bash
   jupyter nbconvert --to script 01a_DataGenerator.ipynb
   nohup python 01a_DataGenerator.py > generation.log 2>&1 &
   ```